In [0]:
"""
ML Experiment: Predict seconds_to_next_stop using Gold layer features.
Final model: Extra Trees Regressor (chosen after comparing 11 models + tuning).
"""

import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from mlflow.models import infer_signature
from datetime import datetime, timezone

catalog = "transit_analytics"
gold_schema = "gold"

# --- Load Gold features ---
vtf = spark.table(f"{catalog}.{gold_schema}.vehicle_trip_features")
route_summary = spark.table(f"{catalog}.{gold_schema}.route_performance_summary") \
    .select("route_id", "avg_speed")

df = (
    vtf.join(route_summary, on="route_id", how="left")
    .select("latitude", "longitude", "speed", "bearing", "avg_speed", "seconds_to_next_stop")
    .dropna()
    .toPandas()
)

X = df.drop(columns=["seconds_to_next_stop"])
y = df["seconds_to_next_stop"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- Train final chosen model ---
mlflow.set_experiment("/Shared/transit_eta_prediction")

with mlflow.start_run(run_name="extra_trees_final"):
    model = ExtraTreesRegressor(
        n_estimators=300,
        max_depth=None,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42,
    )
    model.fit(X_train_scaled, y_train)
    predictions = model.predict(X_test_scaled)

    mae = mean_absolute_error(y_test, predictions)
    r2 = r2_score(y_test, predictions)

    mlflow.log_param("model_type", "ExtraTreesRegressor")
    mlflow.log_param("n_estimators", 300)
    mlflow.log_param("max_depth", "None")
    mlflow.log_param("min_samples_split", 5)
    mlflow.log_param("min_samples_leaf", 2)
    mlflow.log_param("features", list(X.columns))
    mlflow.log_param("training_rows", len(X_train))
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)

    signature = infer_signature(X_train_scaled, model.predict(X_train_scaled))
    mlflow.sklearn.log_model(model, "model", signature=signature)

    print(f"Extra Trees (final) -> MAE: {mae:.2f} seconds, R2: {r2:.3f}")

# --- Write predictions table for dashboard ---
predictions_df = X_test.copy()
predictions_df["actual_seconds_to_next_stop"] = y_test.values
predictions_df["predicted_seconds_to_next_stop"] = predictions
predictions_df["prediction_error"] = (
    predictions_df["predicted_seconds_to_next_stop"] - predictions_df["actual_seconds_to_next_stop"]
)
predictions_df["model_run_ts"] = datetime.now(timezone.utc)

spark_predictions_df = spark.createDataFrame(predictions_df)

spark_predictions_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{catalog}.{gold_schema}.eta_predictions"
)

print("eta_predictions row count:", spark_predictions_df.count())
display(spark_predictions_df.limit(10))